In [1]:
import ydf
import pandas as pd
import numpy as np

In [2]:
crime_dataset = pd.read_csv("./Crime_Data_from_2020_to_2024.csv")
crime_dataset = crime_dataset.drop(columns = ["DR_NO", "Date Rptd", "AREA", "Part 1-2", "Crm Cd", "Mocodes", "Premis Cd", "Weapon Used Cd", "Status Desc", "LOCATION", "Cross Street"])

crime_dataset["DATE OCC"] = pd.to_datetime(crime_dataset["DATE OCC"], format = "%Y %b %d %I:%M:%S %p")
crime_dataset = crime_dataset[crime_dataset["DATE OCC"].dt.year == 2020]
crime_dataset["DATE OCC"] = crime_dataset["DATE OCC"].dt.day_of_year

crime_dataset.loc[crime_dataset["Status"] == "AA", "Status"] = "AR"
crime_dataset.loc[crime_dataset["Status"] == "AO", "Status"] = "OT"
crime_dataset.loc[crime_dataset["Status"] == "JA", "Status"] = "AR"
crime_dataset.loc[crime_dataset["Status"] == "JO", "Status"] = "OT"
crime_dataset.head(20)



,DATE OCC,TIME OCC,AREA NAME,Rpt Dist No,Crm Cd Desc,Vict Age,Vict Sex,Vict Descent,Premis Desc,Weapon Desc,Status,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LAT,LON
0,312,845,N Hollywood,1502,THEFT OF IDENTITY,31,M,H,SINGLE FAMILY DWELLING,NaN,IC,354.0,NaN,NaN,NaN,34.2124,-118.4092
1,292,1845,N Hollywood,1521,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",32,M,H,SIDEWALK,KNIFE WITH BLADE 6INCHES OR LESS,IC,230.0,NaN,NaN,NaN,34.1993,-118.4203
2,304,1240,Van Nuys,933,THEFT OF IDENTITY,30,M,W,SINGLE FAMILY DWELLING,NaN,IC,354.0,NaN,NaN,NaN,34.1847,-118.4509
3,359,1310,Wilshire,782,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,47,F,A,STREET,NaN,IC,331.0,NaN,NaN,NaN,34.0339,-118.3747
4,273,1830,Pacific,1454,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),63,M,H,ALLEY,NaN,IC,420.0,NaN,NaN,NaN,33.9813,-118.4350
5,316,1210,Hollenbeck,429,THEFT OF IDENTITY,35,M,B,"MULTI-UNIT DWELLING (APARTMENT, DUPLEX, ETC)",NaN,IC,354.0,NaN,NaN,NaN,34.0830,-118.1678
6,107,1350,Southwest,396,THEFT OF IDENTITY,21,F,B,SINGLE FAMILY DWELLING,NaN,IC,354.0,NaN,NaN,NaN,34.0100,-118.2900
7,189,1400,Northeast,1133,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,14,F,H,YARD (RESIDENTIAL/BUSINESS),UNKNOWN WEAPON/OTHER WEAPON,OT,812.0,860.0,NaN,NaN,34.1107,-118.2589
8,62,1200,Devonshire,1729,THEFT OF IDENTITY,43,M,W,SINGLE FAMILY DWELLING,NaN,IC,354.0,NaN,NaN,NaN,34.2763,-118.5210
9,245,900,Topanga,2196,THEFT OF IDENTITY,57,M,W,SINGLE FAMILY DWELLING,NaN,IC,354.0,NaN,NaN,NaN,34.1493,-118.5886


In [3]:
# Set missing values to unknown (X)
crime_dataset['Vict Sex'] = crime_dataset['Vict Sex'].replace(np.nan, "X")
crime_dataset['Vict Sex'] = crime_dataset['Vict Sex'].replace("-", "X")
crime_dataset['Vict Sex'] = crime_dataset['Vict Sex'].replace("H", "X")

crime_dataset['Vict Descent'] = crime_dataset['Vict Descent'].replace(np.nan, "X")
crime_dataset['Vict Descent'] = crime_dataset['Vict Descent'].replace("-", "X")

In [4]:
# Remove negative ages
crime_dataset.loc[crime_dataset['Vict Age'] < 0, 'Vict Age'] = np.nan

In [5]:
#Force strings
crime_dataset["Premis Desc"] = (crime_dataset["Premis Desc"].fillna("UNKNOWN").apply(lambda x: str(x)))
crime_dataset["Weapon Desc"] = (crime_dataset["Weapon Desc"].fillna("UNKNOWN").apply(lambda x: str(x)))

In [6]:
def split_dataset(ds, test_ratio):
  test_indices = np.random.rand(len(ds)) < test_ratio
  return ds[~test_indices], ds[test_indices]

crime_train, crime_test = split_dataset(crime_dataset, 0.10)
print(len(crime_train), len(crime_test))

179871 19976


In [7]:
crime_IC = crime_train[crime_train["Status"] == "IC"]
crime_CC = crime_train[crime_train["Status"] == "CC"]
crime_AR = crime_train[crime_train["Status"] == "AR"]
crime_OT = crime_train[crime_train["Status"] == "OT"]

print(len(crime_IC), len(crime_CC), len(crime_AR), len(crime_OT))

136765 0 19203 23903


In [8]:
crime_IC_keep, _ = split_dataset(crime_IC, 0.4) # keep 60% of IC
crime_AR = pd.concat([crime_AR] * 3)  # repeat AR and OT
crime_OT = pd.concat([crime_OT] * 3)
print(len(crime_IC_keep), len(crime_AR), len(crime_OT))

82393 57609 71709


In [9]:
crime_resample = pd.concat([crime_IC_keep, crime_AR, crime_OT])
crime_resample = crime_resample.sample(frac = 1).reset_index(drop=True) # reshuffle
crime_resample.head(20)

,DATE OCC,TIME OCC,AREA NAME,Rpt Dist No,Crm Cd Desc,Vict Age,Vict Sex,Vict Descent,Premis Desc,Weapon Desc,Status,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LAT,LON
0,364,830,Northeast,1136,THEFT OF IDENTITY,31.0,M,W,CYBERSPACE,UNKNOWN,IC,354.0,NaN,NaN,NaN,34.1158,-118.2172
1,254,1700,Pacific,1453,"LETTERS, LEWD - TELEPHONE CALLS, LEWD",41.0,F,A,PARKING LOT,UNKNOWN,OT,956.0,NaN,NaN,NaN,33.9906,-118.4391
2,281,1200,Hollywood,645,INTIMATE PARTNER - SIMPLE ASSAULT,37.0,M,B,"MULTI-UNIT DWELLING (APARTMENT, DUPLEX, ETC)","STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",OT,626.0,NaN,NaN,NaN,34.1006,-118.3417
3,18,1130,77th Street,1253,VIOLATION OF COURT ORDER,61.0,F,B,SINGLE FAMILY DWELLING,UNKNOWN,OT,900.0,NaN,NaN,NaN,33.9700,-118.3068
4,38,2040,Devonshire,1717,FAILURE TO YIELD,30.0,M,W,STREET,UNKNOWN,AR,890.0,NaN,NaN,NaN,34.2758,-118.4934
5,35,2220,Southeast,1802,VIOLATION OF COURT ORDER,42.0,F,B,SINGLE FAMILY DWELLING,UNKNOWN,IC,900.0,NaN,NaN,NaN,33.9551,-118.2782
6,322,1620,Mission,1971,BATTERY - SIMPLE ASSAULT,12.0,M,H,SIDEWALK,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",IC,624.0,NaN,NaN,NaN,34.2337,-118.4703
7,131,1200,Northeast,1162,INTIMATE PARTNER - SIMPLE ASSAULT,25.0,F,B,HOSPITAL,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",OT,626.0,NaN,NaN,NaN,34.0956,-118.2918
8,103,1400,Devonshire,1782,BRANDISH WEAPON,27.0,M,W,SIDEWALK,UNKNOWN FIREARM,OT,761.0,NaN,NaN,NaN,34.2291,-118.5634
9,63,200,Devonshire,1762,"LETTERS, LEWD - TELEPHONE CALLS, LEWD",50.0,M,O,SINGLE FAMILY DWELLING,UNKNOWN,OT,956.0,NaN,NaN,NaN,34.2452,-118.5763


In [10]:
crime_resample.value_counts("Status")

Status
IC    82393
OT    71709
AR    57609
Name: count, dtype: int64

In [ ]:
#crime_final_train, _ = split_dataset(crime_resample, 0.93) # pare down for testing, REMOVE FOR FINAL PASS
#crime_final_test, _ = split_dataset(crime_test, 0.93)
#print(len(crime_final_train), len(crime_final_test))

In [11]:
model = ydf.RandomForestLearner(label = "Status", num_trees= 500).train(crime_resample)

Train model on 211711 examples
Model trained in 0:00:20.264095


In [12]:
model.evaluate(crime_test)

Label \ Pred,IC,OT,AR
IC,12032,556,584
OT,2233,1755,793
AR,877,371,775


In [13]:
model.analyze(crime_test)